# Phase 5 · data statistics and a no-language-model baseline (CPU runtime is enough)
Run this first. It prints document lengths in tokens, statutes per document and how common each statute is, then scores **TF-IDF + a one-vs-rest linear model** on dev. The language-model runs have to beat that to show the Phase 1 model adds anything. It also compares one global threshold with per-statute thresholds, with the per-statute thresholds scored honestly (tuned on one half of dev, scored on the other).

In [ ]:
# 1 · Settings
from pathlib import Path
SEED, N_TRAIN, N_DEV, MAX_TOKENS = 17, 12000, 3000, 40000
COURSE_ROOT = Path('/content/drive/MyDrive/legal-llm-course')
PHASE_DIR   = COURSE_ROOT/'Phase_5_LSI'
BUNDLE_ZIP  = COURSE_ROOT/'phase2_lner_multimethod_bundle.zip'
RR_LIB, CJPE_LIB, LSI_LIB = COURSE_ROOT/'rr_lib.py', COURSE_ROOT/'cjpe_lib.py', COURSE_ROOT/'lsi_lib.py'

In [ ]:
# 2 · Drive, helper bundle, tokenizer, device
from google.colab import drive
drive.mount('/content/drive')
import sys, shutil, zipfile, hashlib, json, subprocess, importlib.util, random
LOCAL = Path('/content/phase3-rr')
if LOCAL.exists(): shutil.rmtree(LOCAL)
LOCAL.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE_ZIP) as z: z.extractall(LOCAL)
shutil.copy(RR_LIB, LOCAL/'rr_lib.py'); shutil.copy(CJPE_LIB, LOCAL/'cjpe_lib.py'); shutil.copy(LSI_LIB, LOCAL/'lsi_lib.py')
sys.path.insert(0, str(LOCAL))
need = [p for p in ('tokenizers','huggingface_hub','scikit-learn') if importlib.util.find_spec(p.replace('scikit-learn','sklearn')) is None]
if need: subprocess.check_call([sys.executable,'-m','pip','install','-q',*need])
import torch, rr_lib as R, cjpe_lib as J, lsi_lib as LSI
from tokenizers import Tokenizer
from training_methods import resolve_snapshot_checkpoint
TOKENIZER_SHA256 = '906194848c1632431aac7d186e7c4f49da83cb6cbd480fa222c8472401cbf8ab'
tp = LOCAL/'tokenizer/tokenizer.json'
assert hashlib.sha256(tp.read_bytes()).hexdigest() == TOKENIZER_SHA256, 'tokenizer differs from Phase 1/2'
TOK = Tokenizer.from_file(str(tp))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); torch.manual_seed(SEED)
import numpy as np
print('device', device, torch.cuda.get_device_name(0) if device.type=='cuda' else '')

In [ ]:
# 3 · Data (downloads Phase_5_LSI from the gated HF dataset if not already on Drive) + statistics
import pandas as pd
from google.colab import userdata
try: token = userdata.get('HF_TOKEN')
except Exception: token = None
LSI.ensure_files(COURSE_ROOT, token)
TEXT = LSI.load_all(PHASE_DIR)
for s, docs in TEXT.items(): print(f'{s}: {len(docs)} documents')
sample = TEXT['train'][:3000]; J.tokenize_docs(sample, TOK, MAX_TOKENS); n = np.array([len(d['tok']) for d in sample])
print('tokens per document (first 3,000 train): mean/median/p90/max', int(n.mean()), int(np.median(n)), int(np.percentile(n, 90)), int(n.max()), '| 512-token chunks: mean', round(np.mean([len(J.chunk_spans(len(d['tok']), 512, 512)) for d in sample]), 1), '| documents fully covered by 8 chunks:', f"{100*np.mean([len(J.chunk_spans(len(d['tok']), 512, 512)) <= 8 for d in sample]):.0f}%")
Ytr_all = LSI.multi_hot(TEXT['train']); freq = Ytr_all.sum(0)
print('statutes per document (train): mean', round(Ytr_all.sum(1).mean(), 2), '| documents per statute: min', int(freq.min()), 'median', int(np.median(freq)), 'max', int(freq.max()))

In [ ]:
# 4 · TF-IDF + one-vs-rest LinearSVC baseline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
rng = np.random.RandomState(SEED); tr = [TEXT['train'][i] for i in rng.permutation(len(TEXT['train']))[:N_TRAIN]]; dv = [TEXT['dev'][i] for i in rng.permutation(len(TEXT['dev']))[:N_DEV]]
ytr, ydv = LSI.multi_hot(tr), LSI.multi_hot(dv)
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=200000, sublinear_tf=True, dtype=np.float32)
A = vec.fit_transform([d['text'] for d in tr]); B = vec.transform([d['text'] for d in dv]); print('tf-idf', A.shape)
rows = []
for C_ in (0.1, 0.5):
    S = OneVsRestClassifier(LinearSVC(C=C_), n_jobs=-1).fit(A, ytr).decision_function(B); P = 1 / (1 + np.exp(-4 * S))     # squash scores to (0,1) so the threshold grid applies
    ev = LSI.evaluate_probs(ydv, P)
    for k in ('fixed_0.5', 'global_tuned', 'global_tuned+min_one', 'per_label_cross_fit+min_one'):
        rows.append({'C': C_, 'threshold strategy': k, **{m: round(ev[k][m], 4) for m in ('macro_f1', 'micro_f1', 'subset_acc')}})
display(pd.DataFrame(rows))
f1 = np.array(LSI.scores(ydv, LSI.decide(P, ev['global_threshold'], True))['per_label_f1']); order = np.argsort(f1)
print('worst 5 statutes (id, F1):', [(int(k), round(float(f1[k]), 2)) for k in order[:5]], '| best 5:', [(int(k), round(float(f1[k]), 2)) for k in order[-5:]])